In [1]:
%%writefile ../src/features.py
"""Which columns may be used to predict lateness, and the preprocessing pipeline."""
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src import config

# Known only after the shipment happens, or computed from the outcome itself
OUTCOME_COLS = [
    config.REAL_DAYS, config.GAP, config.DELIVERY_STATUS,
    config.DELIVERY_CLASS, config.IS_CANCELED,
]

# Not known at order time, or net of costs that arise during fulfilment.
# Excluded to be safe; can be revisited if the business confirms timing.
EXCLUDED_UNSURE_COLS = [
    "Order Status",                      # changes after the order is placed
    "Benefit per order", "Order Profit Per Order", "Order Item Profit Ratio",
]

# Duplicates of other information (each checked against the data)
REDUNDANT_COLS = [
    config.SCHED_DAYS,                   # fully determined by Shipping Mode
    "Category Id", "Department Id",      # same as the name columns
    "Sales per customer",                # equals Order Item Total on 100% of rows
    "Product Price",                     # equals Order Item Product Price on 100% of rows
    "Sales",                             # equals price x quantity on 100% of rows
]

# Identifiers (a model should not memorise individuals)
ID_COLS = ["Customer Id"]

# Too many levels for a sparse, mostly-small-group column (3,488 / 1,062 / 563
# levels in train). Geography is still covered by Market, Region, Country, State.
HIGH_CARDINALITY_DROP = ["Order City", "Order State", "Customer City"]

NEVER_USE = ([config.TARGET] + OUTCOME_COLS + EXCLUDED_UNSURE_COLS
             + REDUNDANT_COLS + ID_COLS + HIGH_CARDINALITY_DROP)

MIN_CATEGORY_COUNT = 100   # categories rarer than this are pooled into "infrequent"


def candidate_features(df: pd.DataFrame) -> list:
    """Columns left after removing everything above."""
    return [c for c in df.columns if c not in NEVER_USE]


def split_feature_types(df: pd.DataFrame):
    feats = candidate_features(df)
    numeric = df[feats].select_dtypes(include="number").columns.tolist()
    categorical = [c for c in feats if c not in numeric]
    return numeric, categorical


def build_preprocessor(numeric, categorical) -> ColumnTransformer:
    """Imputers, scalers and encoders. Fitted only inside a Pipeline, so in
    cross-validation they learn from the training folds alone."""
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=MIN_CATEGORY_COUNT)),
    ])
    return ColumnTransformer([("num", num_pipe, numeric),
                              ("cat", cat_pipe, categorical)])

Overwriting ../src/features.py


In [2]:
import sys
sys.path.append("..")
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from src import config
from src.features import split_feature_types, build_preprocessor, NEVER_USE
from src.evaluate import fold_scores

train = pd.read_parquet(config.ROOT / "data/processed/train.parquet")
numeric, categorical = split_feature_types(train)

print(len(numeric), "numeric    :", numeric)
print(len(categorical), "categorical:", categorical)
assert not set(numeric + categorical) & set(NEVER_USE), "forbidden column slipped in!"
print("Leakage guard passed")

7 numeric    : ['Latitude', 'Longitude', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Order Item Total']
11 categorical: ['Type', 'Category Name', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order Country', 'Order Region', 'Product Name', 'Shipping Mode']
Leakage guard passed


In [3]:
features = numeric + categorical
X, y, groups = train[features], train[config.TARGET], train["Customer Id"]

pre = build_preprocessor(numeric, categorical)
Xt = pre.fit_transform(X)
print("Encoded matrix:", Xt.shape, type(Xt).__name__)

Encoded matrix: (138213, 306) csr_matrix


In [4]:
logreg = Pipeline([("prep", build_preprocessor(numeric, categorical)),
                   ("clf", LogisticRegression(max_iter=1000))])

f = fold_scores(logreg, X, y, groups)
pd.DataFrame({"mean": f.mean(), "std": f.std()}).round(4)

,mean,std
roc_auc,0.7390,0.0054
pr_auc,0.8327,0.0035
log_loss,0.5406,0.0035
brier,0.1896,0.0015
accuracy@0.5,0.6942,0.0046
precision@0.5,0.8585,0.0089
recall@0.5,0.5583,0.0075


In [5]:
p = train.groupby(config.SHIP_MODE)[config.TARGET].transform("mean").values
rng = np.random.default_rng(0)

print("Mode-only PR-AUC, ties as-is        :", round(average_precision_score(y, p), 4))
print("Mode-only PR-AUC, ties broken by noise:",
      round(average_precision_score(y, p + rng.normal(0, 1e-6, len(p))), 4))

Mode-only PR-AUC, ties as-is        : 0.784
Mode-only PR-AUC, ties broken by noise: 0.8347
